# Week 9 Homework: Building and Comparing FAQ Search Engines


## Purpose of Homework

In this homework you will build and compare four search engines that answer student questions using the University of Toronto Department of Statistical Sciences FAQ page. Each search engine uses a different text representation — raw term frequency, TF-IDF, and two autoencoder bottleneck embeddings — to find the most relevant FAQ entry for a given query.

You will practise:
- Understanding web scraping code
- Building document-term matrices with `CountVectorizer` and `TfidfVectorizer`
- Training shallow autoencoders in Keras
- Using cosine similarity to retrieve relevant documents
- Evaluating and comparing search engines quantitatively and qualitatively

## Logistics

Due date: The homework is due **11:59pm on Thursday, April 2, 2026.**

You will submit your homework on [MarkUs](https://markus.teach.cs.toronto.edu/markus/).

1. Download this file (`STA272_hw9_student.ipynb`) from JupyterHub. (See [our JupyterHub Guide](../guides/jupyterhub_guide.ipynb) for detailed instructions.)
2. Submit this file to MarkUs under the hw9 assignment. (See [our MarkUs Guide](../guides/markus_guide.ipynb) for detailed instructions.)

## Data: UofT Statistics FAQ

The file `stats_faqs.json` contains frequently asked questions from the Department of Statistical Sciences website. Each entry has a `"question"` and an `"answer"` field.

**Run the setup cell below. Do not modify it.**

In [ ]:
# ---- Setup (run this cell, do not modify) ----
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

# Load FAQ data from JSON (same pattern as Week 2)
faq_df = pd.read_json('stats_faqs.json')
questions = faq_df['question'].tolist()
answers = faq_df['answer'].tolist()

print(f"{len(questions)} FAQ entries loaded")
faq_df.head()

## Task #1: Exploring Word Frequencies

**(a)** Combine all the FAQ questions into a single string, tokenise it (lowercase, letters only), and compute the frequency of each word using `pd.Series.value_counts()`. Store the result as a Series called `word_freq`.

**(b)** Plot a horizontal bar chart of the **top 20 most frequent words**.

**(c)** In the markdown cell below the plot, list **at least 5 words** from the top 20 that you think should be treated as stop words for this FAQ dataset. Explain briefly why these words would not help a search engine distinguish between different FAQ topics.

In [ ]:
# Place your answer for Task #1 (a) and (b) in this cell
import re

# (a) Tokenise and count
all_text = ' '.join(questions)
words = re.findall(r'[a-z]+', all_text.lower())
word_freq = ...

print(f"Total words: {len(words)}  |  Unique words: {len(word_freq)}")
print(word_freq.head(20))

# (b) Bar chart of top 20
...

**(c)** List at least 5 words from the top 20 that should be treated as stop words for this FAQ dataset. Why would these words not help a search engine distinguish between FAQ topics?

*Your answer for (c) here*

## Task #2: Term Frequency Search Engine

Build a search engine using **raw term counts** (no IDF weighting).

1. Use `CountVectorizer` from sklearn with `stop_words='english'` to build a term-frequency matrix from `questions`.
2. Write a function `search_tf(query, top_k=3)` that:
   - Transforms the query using the **same** vectorizer
   - Computes cosine similarity against all FAQ questions
   - Returns a DataFrame with columns `['question', 'answer', 'similarity']`, sorted by similarity (highest first)

Store:
- The fitted vectorizer as `cv`
- The term-frequency matrix as `tf_matrix`

Test your function with the query `"How do I switch my program?"` and store the result as `tf_test`.

In [ ]:
# Place your answer for Task #2 in this cell
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(...)
tf_matrix = ...

def search_tf(query, top_k=3):
    query_vec = cv.transform(...)
    sims = cosine_similarity(...)[0]
    best_idx = ...
    results = []
    for i in best_idx:
        results.append({
            'question': questions[i],
            'answer': answers[i],
            'similarity': round(sims[i], 4)
        })
    return pd.DataFrame(results)

tf_test = search_tf("How do I switch my program?")
print(tf_test)

## Task #3: TF-IDF Search Engine

Build a search engine using **TF-IDF** weighting.

1. Use `TfidfVectorizer` with `stop_words='english'` and `max_features=500`.
2. Write a function `search_tfidf(query, top_k=3)` with the same signature and return format as `search_tf`.

Store:
- The fitted vectorizer as `tfidf_vec`
- The TF-IDF matrix as `tfidf_matrix`

Test with the same query and store the result as `tfidf_test`.

In [ ]:
# Place your answer for Task #3 in this cell
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec = TfidfVectorizer(...)
tfidf_matrix = ...

def search_tfidf(query, top_k=3):
    ...

tfidf_test = search_tfidf("How do I switch my program?")
print(tfidf_test)

## Task #4: Autoencoder Search Engine (16-dim)

Build a search engine using a **16-dimensional autoencoder bottleneck** on top of the TF-IDF representation.

1. Convert `tfidf_matrix` to a dense float32 array (the autoencoder needs dense input).
2. Build a shallow autoencoder in Keras:
   - Input layer: shape = number of TF-IDF features
   - Bottleneck: `Dense(16, activation='relu')` — name this layer `'bottleneck'`
   - Output: `Dense(n_features, activation='sigmoid')`
3. Compile with `optimizer='adam'`, `loss='mse'`.
4. Fit on the TF-IDF matrix for `epochs=100`, `batch_size=8`, `verbose=0`.
5. Extract the encoder (input → bottleneck) and compute embeddings for all FAQ questions. Store as `ae16_matrix`.
6. Write `search_ae16(query, top_k=3)` that: TF-IDF transforms the query → encodes through the bottleneck → cosine similarity against `ae16_matrix`.

Store: `autoencoder_16`, `encoder_16`, `ae16_matrix`

Test with the same query and store as `ae16_test`.

In [ ]:
# Place your answer for Task #4 in this cell
from tensorflow import keras
from tensorflow.keras import layers

X_dense = tfidf_matrix.toarray().astype('float32')
n_features = X_dense.shape[1]

# Build autoencoder
inp = keras.Input(shape=(n_features,))
bottleneck = layers.Dense(..., activation=..., name='bottleneck')(inp)
output = layers.Dense(..., activation=...)(bottleneck)

autoencoder_16 = keras.Model(...)
autoencoder_16.compile(...)
autoencoder_16.fit(...)

# Extract encoder
encoder_16 = keras.Model(...)
ae16_matrix = ...

def search_ae16(query, top_k=3):
    ...

ae16_test = search_ae16("How do I switch my program?")
print(ae16_test)

## Task #5: Autoencoder Search Engine (32-dim)

Repeat Task #4 with a **32-dimensional bottleneck**. Name the bottleneck layer `'bottleneck_32'`.

Store: `autoencoder_32`, `encoder_32`, `ae32_matrix`

Write `search_ae32(query, top_k=3)` and test with the same query. Store as `ae32_test`.

In [ ]:
# Place your answer for Task #5 in this cell

...

ae32_test = search_ae32("How do I switch my program?")
print(ae32_test)

## Task #6: Quantitative Comparison

Evaluate all four search engines using a **self-retrieval test**: for each FAQ question, use it as a query and check whether the search engine returns **that same question** as the top-1 result.

For each method, compute:
- `mean_top1_sim`: the average cosine similarity of the top-1 result across all queries
- `self_retrieval_rate`: the fraction of queries where the top-1 result is the query question itself

Store the results as `comparison_df` — a DataFrame with columns `['method', 'mean_top1_sim', 'self_retrieval_rate']` and 4 rows (one per method: `'Term Frequency'`, `'TF-IDF'`, `'Autoencoder 16'`, `'Autoencoder 32'`).

*Hint:* for each question `i`, call the search function with `top_k=1` and check if the returned question matches `questions[i]`.

In [ ]:
# Place your answer for Task #6 in this cell

methods = {
    'Term Frequency': search_tf,
    'TF-IDF': search_tfidf,
    'Autoencoder 16': search_ae16,
    'Autoencoder 32': search_ae32
}

rows = []
for name, search_fn in methods.items():
    top1_sims = []
    self_matches = 0
    for i, q in enumerate(questions):
        result = search_fn(q, top_k=1)
        ...
    rows.append({
        'method': name,
        'mean_top1_sim': ...,
        'self_retrieval_rate': ...
    })

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))

## Task #7: Qualitative Comparison

Choose **2 queries** of your own — realistic questions a student might ask about the statistics program (e.g., about courses, switching programs, prerequisites, graduation requirements).

Run all 4 search engines on each query (with `top_k=1`) and collect the results.

Store:
- `query1` — your first query string
- `query2` — your second query string
- `query1_results` — a DataFrame with columns `['method', 'top_question', 'top_answer', 'similarity']` (4 rows)
- `query2_results` — same format

In [ ]:
# Place your answer for Task #7 in this cell

query1 = "..."
query2 = "..."

def compare_all(query):
    rows = []
    for name, search_fn in methods.items():
        result = search_fn(query, top_k=1)
        rows.append({
            'method': name,
            'top_question': result['question'].iloc[0],
            'top_answer': result['answer'].iloc[0][:200],
            'similarity': result['similarity'].iloc[0]
        })
    return pd.DataFrame(rows)

query1_results = compare_all(query1)
query2_results = compare_all(query2)

print(f"Query 1: {query1}")
print(query1_results[['method', 'top_question', 'similarity']].to_string(index=False))
print(f"\nQuery 2: {query2}")
print(query2_results[['method', 'top_question', 'similarity']].to_string(index=False))

## Task #8: Interpretation — Which Search Engine is Best? (not autograded)

Write a short comparison (~200 words) addressing the following:

1. Which method had the highest **self-retrieval rate** in Task #6, and why?
2. For your two custom queries in Task #7, which method returned the **most helpful** answer? Did all methods agree?
3. What are the trade-offs between TF-IDF (sparse, interpretable) and autoencoder embeddings (dense, learned)?
4. Would you expect the 32-dim autoencoder to **always** outperform the 16-dim one? Why or why not?

*Your answer here*

## Question #1: Bias in Word Embeddings

In lecture we saw that word embeddings like GloVe encode meaning by placing semantically similar words close together in vector space. A famous example is the analogy **man : woman :: king : queen**, which can be recovered via vector arithmetic: `king - man + woman ≈ queen`.

However, because embeddings are trained on large text corpora (e.g., Wikipedia, news articles), they also absorb the **implicit biases and stereotypes** present in that text. For example, researchers have found that the same vector arithmetic produces analogies like **man : computer programmer :: woman : homemaker** (Bolukbasi et al., 2016).

**(a)** Explain in your own words *why* embeddings trained on text data can encode societal biases. What is the mechanism — what property of the training data leads to biased embeddings? (2 marks)

**(b)** Suppose you build a FAQ search engine for a university admissions office using embeddings trained on historical text. Give a concrete example of how embedding bias could lead to **unfair or misleading search results** for certain users. (2 marks)

**(c)** The TF-IDF representation does not use pre-trained embeddings. Is TF-IDF immune to bias? Why or why not? (1 mark)

*Your answer here*

## Question #2: Stop Words and IDF

**(a)** Why do we remove stop words before building the TF-IDF matrix? (1 mark)

**(b)** Wouldn’t IDF already down-weight very common words? Give a concrete example of a word that IDF would **not** down-weight sufficiently, and explain why explicit removal is still useful. (2 marks)

*Your answer here*

## Question #3: Autoencoder Bottleneck Trade-off

**(a)** The 16-dim autoencoder compresses ~500 TF-IDF weights into 16 numbers. What kind of information might be lost in this compression? (1 mark)

**(b)** Despite this information loss, when would you prefer the 16-dim bottleneck embedding over the full TF-IDF vector for a downstream task? (2 marks)

*Your answer here*